# Meyer Penny Game

In [6]:
import pyquil.quil as pq
from pyquil.api import get_qc
from pyquil.gates import I, H, X

import numpy as np

In [7]:
def meyer_penny_program():
    """
    Returns the program to simulate the Meyer-Penny Game
    The full description is available in ../docs/source/exercises.rst

    :return: pyQuil Program
    """
    prog = pq.Program()
    ro = prog.declare("ro", memory_size=2)
    picard_register = ro[1]
    answer_register = ro[0]

    then_branch = pq.Program(X(0))
    else_branch = pq.Program(I(0))

    # Prepare Qubits in Heads state or superposition, respectively
    prog.inst(X(0), H(1))
    # Q puts the coin into a superposition
    prog.inst(H(0))
    # Picard makes a decision and acts accordingly
    prog.measure(1, picard_register)
    prog.if_then(picard_register, then_branch, else_branch)
    # Q undoes his superposition operation
    prog.inst(H(0))
    # The outcome is recorded into the answer register
    prog.measure(0, answer_register)

    return prog

In [11]:
n_trials = 10
qvm = get_qc("2q-qvm")

prog = meyer_penny_program()
print(prog)
prog.resolve_label_placeholders()      # turn if_then's placeholder labels into real labels
prog.wrap_in_numshots_loop(n_trials)
outcomes = qvm.run(prog).get_register_map()["ro"]   # (n_trials, 2) array

print("Number of games: {}".format(n_trials))
print("Q's winning average: {}".format(outcomes[:, 0].mean()))


DECLARE ro BIT[2]
X 0
H 1
H 0
MEASURE 1 ro[1]
JUMP-WHEN @Placeholder(TargetPlaceholder("THEN")) ro[1]
I 0
JUMP @Placeholder(TargetPlaceholder("END"))
LABEL @Placeholder(TargetPlaceholder("THEN"))
X 0
LABEL @Placeholder(TargetPlaceholder("END"))
H 0
MEASURE 0 ro[0]

Number of games: 10
Q's winning average: 1.0
